# Step 5: Model Evaluation

This notebook evaluates the trained Patient Risk Classification model by:

1. Loading the model from the **Snowflake Model Registry**
2. Running inference on the held-out test set
3. Computing multiclass metrics (accuracy, precision, recall, F1, log loss)
4. Logging results to a `MODEL_METRICS` table for downstream promotion decisions

In [ ]:
%cd ..
%load_ext autoreload

## Imports and Configuration

Connect to Snowflake, load pipeline config, and set up the database/schema/warehouse context.

In [ ]:
import os
import sys
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

from snowflake.snowpark import Session
from source.configs import get_config
from source.utils import get_session, get_feature_config

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
COMPUTE_WAREHOUSE = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

## Perform Inference

Load the latest model version from the **Snowflake Model Registry**, then run inference on the held-out test set:

1. **Get model version** — retrieve the latest registered version of `PATIENT_RISK_MODEL`
2. **Load test data** — read from the `TEST_FEATURES` table created during preprocessing
3. **Run predictions** — call `model_version.run()` with `predict` for class labels and `predict_proba` for probability scores

In [ ]:
from snowflake.ml.registry import Registry

model_name = "PATIENT_RISK_MODEL"
test_table = f"{DB}.{SCHEMA}.TEST_FEATURES"
feature_config = get_feature_config(config)
numeric_columns = feature_config["all_numeric_features"]
categorical_columns = feature_config["all_categorical_features"]
feature_columns = numeric_columns + categorical_columns
target_column = feature_config["target_column"]

# == Get Model from Registry ==
registry = Registry(
            session,
            database_name=DB,
            schema_name=SCHEMA,
        )
model = registry.get_model(model_name)
model_version_obj = model.versions()[-1]


# == Get Test Data ==
logger.info(f"Loading test data from {test_table}")
test_df = session.table(test_table).to_pandas()
test_df.columns = [c.upper() for c in test_df.columns]
features = test_df[feature_columns]
y_true = test_df[target_column].values


logger.info("Running inference")
predictions_df = model_version_obj.run(features, function_name="predict")
y_pred = predictions_df["output_feature_0"].values


proba_df = model_version_obj.run(features, function_name="predict_proba")
y_proba = proba_df.values

## Compute Evaluation Metrics

Define a function to compute a comprehensive set of multiclass classification metrics:

- **Macro-averaged** metrics (precision, recall, F1): unweighted mean across all classes — treats each class equally regardless of size
- **Weighted-averaged** metrics: weighted by class frequency — accounts for class imbalance
- **Log loss**: measures prediction confidence using probability outputs
- **Per-class** metrics: precision, recall, F1, and support for each risk level (LOW, MEDIUM, HIGH, CRITICAL)
- **Confusion matrix**: shows where the model confuses one class for another

In [ ]:
import numpy as np
from typing import Optional, List, Dict, Any
from datetime import datetime

def compute_multiclass_metrics(
        y_true: np.ndarray,
        y_pred: np.ndarray,
        y_proba: Optional[np.ndarray] = None,
        class_labels: Optional[List[str]] = None,
    ) -> Dict[str, Any]:
        from sklearn.metrics import (
            accuracy_score,
            confusion_matrix,
            f1_score,
            log_loss,
            precision_score,
            recall_score,
        )

        # Overall accuracy
        accuracy = accuracy_score(y_true, y_pred)

        # Macro-averaged metrics (unweighted mean across classes)
        precision_macro = precision_score(
            y_true, y_pred, average="macro", zero_division=0
        )
        recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
        f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

        # Weighted-averaged metrics (weighted by class frequency)
        precision_weighted = precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        )
        recall_weighted = recall_score(
            y_true, y_pred, average="weighted", zero_division=0
        )
        f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)

        metrics = {
            "accuracy": float(accuracy),
            "precision_macro": float(precision_macro),
            "recall_macro": float(recall_macro),
            "f1_macro": float(f1_macro),
            "precision_weighted": float(precision_weighted),
            "recall_weighted": float(recall_weighted),
            "f1_weighted": float(f1_weighted),
            "confusion_matrix": cm.tolist(),
        }

        # Log loss if probabilities are available
        if y_proba is not None:
            try:
                logloss = log_loss(y_true, y_proba)
                metrics["log_loss"] = float(logloss)
            except Exception as e:
                logger.warning(f"Could not compute log loss: {e}")

        # Per-class metrics if labels are provided
        if class_labels:
            per_class_metrics = {}
            for _, label in enumerate(class_labels):
                # Convert to binary classification for this class
                y_true_binary = (y_true == label).astype(int)
                y_pred_binary = (y_pred == label).astype(int)

                per_class_metrics[label] = {
                    "precision": float(
                        precision_score(y_true_binary, y_pred_binary, zero_division=0)
                    ),
                    "recall": float(
                        recall_score(y_true_binary, y_pred_binary, zero_division=0)
                    ),
                    "f1": float(
                        f1_score(y_true_binary, y_pred_binary, zero_division=0)
                    ),
                    "support": int(np.sum(y_true == label)),
                }

            metrics["per_class"] = per_class_metrics

        logger.info(
            f"Computed metrics: accuracy={accuracy:.4f}, f1_macro={f1_macro:.4f}"
        )
        metrics["test_size"] = len(y_pred)
        metrics["evaluated_at"] = datetime.now().isoformat()

        return metrics

metrics = compute_multiclass_metrics(
            y_true=y_true,
            y_pred=y_pred,
            y_proba=y_proba,
            class_labels=feature_config["class_labels"],
        )


# Plot Confusion Matrix

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = np.array(metrics["confusion_matrix"])
class_labels = feature_config["class_labels"]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_labels, yticklabels=class_labels, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — Patient Risk Classification")
plt.tight_layout()
plt.show()

## Log Metrics to Database

Write the computed metrics to a `MODEL_METRICS` table so they can be used for promotion decisions in the deployment step. Each scalar metric (accuracy, precision, recall, F1, etc.) is stored as a separate row with a unique ID and timestamp.

In [ ]:
import uuid
import pandas as pd

records = []
metrics_table = f"{DB}.{SCHEMA}.MODEL_METRICS"
for metric_name, metric_value in metrics.items():
    if metric_name in ["confusion_matrix", "per_class", "evaluated_at"]:
        continue

    if isinstance(metric_value, (int, float)):
        records.append(
            {
                "METRIC_ID": f"M{uuid.uuid4().hex[:8].upper()}",
                "MODEL_NAME": model_name or metrics.get("model_name"),
                "MODEL_VERSION": model_version_obj.version_name,
                "METRIC_NAME": metric_name,
                "METRIC_VALUE": float(metric_value),
                "METRIC_DETAILS": None,
                "EVALUATED_AT": datetime.now(),
            }
        )

if records:
    df = pd.DataFrame(records)
    snowpark_df = session.create_dataframe(df)
    snowpark_df.write.mode("append").save_as_table(metrics_table)
    logger.info(f"Logged {len(records)} metrics to {metrics_table}")

session.table(metrics_table).to_pandas()

---

## Next Steps

Proceed to **[06_model_deployment.ipynb](./06_model_deployment.ipynb)** to deploy the model for SQL inference and create an SPCS inference service.